# 🛒 E-Mall: SASRec Recommendation System
## Graduation Project — GP-02

This notebook covers the full training, evaluation, visualisation, and export pipeline
for the **Self-Attentive Sequential Recommendation (SASRec)** model built for the
E-Mall Smart E-Commerce platform.

### Contents
| # | Section |
|---|----------|
| 0 | 🔗 Google Drive Mount & Config |
| 1 | Setup & Imports |
| 2 | Data Preprocessing |
| 3 | SASRec Architecture |
| 4 | Dataset & DataLoader |
| 5 | Training Loop |
| 6 | ✅ Structured Results Display |
| 7 | 📊 Visualisations & Model Statistics |
| 8 | 🔍 Inference |
| 9 | 💾 Model Export & Reload Test |
| 10 | 🌐 API Server (Postman / curl ready) |

## 0. 🔗 Google Drive Mount & Path Configuration

Mount your Google Drive and set `BASE_PATH` to the folder containing
`interactions.csv` and the `api/` directory.

> **Note:** Leave `DRIVE_SUBFOLDER` empty (`""`) if the project sits at the root of your Drive,
> or set it to e.g. `"GP - Dataset"` if you placed the folder inside Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure your project path ───────────────────────────────
# Change DRIVE_SUBFOLDER to match where the project is in Drive.
# Leave empty ("") if the files are at the root of My Drive.
DRIVE_SUBFOLDER = ""   # e.g. "GP - Dataset"  or  "projects/GP-02"

import os
if DRIVE_SUBFOLDER:
    BASE_PATH = os.path.join('/content/drive/MyDrive', DRIVE_SUBFOLDER)
else:
    BASE_PATH = '/content/drive/MyDrive'

# Verify the key files exist
assert os.path.isfile(os.path.join(BASE_PATH, 'interactions.csv')), \
    f"interactions.csv not found in {BASE_PATH}. Check DRIVE_SUBFOLDER."

print(f"BASE_PATH = {BASE_PATH}")
print(f"Files: {os.listdir(BASE_PATH)}")

## 1. Setup & Imports

In [ ]:
import os, random, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

# ── Visualisation ────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
matplotlib.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
FIGURES_DIR = os.path.join(BASE_PATH, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Reproducibility ──────────────────────────────────────────────
def set_seed(seed: int = 42):
    """Ensure deterministic behaviour across all libraries."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("  ENVIRONMENT SETUP")
print("=" * 50)
print(f"  Device        : {DEVICE}")
print(f"  PyTorch       : {torch.__version__}")
print(f"  NumPy         : {np.__version__}")
print(f"  Base Path     : {BASE_PATH}")
print(f"  Figures dir   : {os.path.abspath(FIGURES_DIR)}")
print("=" * 50)

## 2. Data Preprocessing

Loads `interactions.csv` (E-Mall DB schema), builds chronological user sequences,
and splits them using the **leave-one-out** strategy:

| Split | Items used |
|-------|------------|
| Train | All items except last 2 |
| Val   | 2nd-to-last item |
| Test  | Last item |

In [ ]:
MAX_SEQ_LEN = 50

print("=" * 50)
print("  DATA PREPROCESSING")
print("=" * 50)

# ── Load ─────────────────────────────────────────────────────────
t0 = time.time()
interactions_df = pd.read_csv(os.path.join(BASE_PATH, "interactions.csv"))

# Handle both 'timestamp' and 'created_at' column names (schema v2)
ts_col = "created_at" if "created_at" in interactions_df.columns else "timestamp"
interactions_df[ts_col] = pd.to_datetime(interactions_df[ts_col])
interactions_df = interactions_df.sort_values(["user_id", ts_col])

print(f"  Total interactions  : {len(interactions_df):>10,}")
print(f"  Unique users        : {interactions_df['user_id'].nunique():>10,}")
print(f"  Unique products     : {interactions_df['product_id'].nunique():>10,}")

# ── Item mappings ─────────────────────────────────────────────────
item_ids  = sorted(interactions_df["product_id"].unique())
item2idx  = {pid: idx for idx, pid in enumerate(item_ids, start=1)}
idx2item  = {idx: pid for pid, idx in item2idx.items()}
NUM_ITEMS = len(item2idx) + 1  # 0 reserved for padding

# ── Sequences ─────────────────────────────────────────────────────
interactions_df["item_idx"] = interactions_df["product_id"].map(item2idx)
user_sequences = (
    interactions_df.groupby("user_id")["item_idx"].apply(list).to_dict()
)

train_data, val_data, test_data = {}, {}, {}
for user, seq in user_sequences.items():
    if len(seq) < 3:
        train_data[user] = seq
        continue
    train_data[user] = seq[:-2]
    val_data[user]   = [seq[-2]]
    test_data[user]  = [seq[-1]]

elapsed = time.time() - t0
print(f"  Mapped items        : {NUM_ITEMS - 1:>10,}")
print(f"  Training users      : {len(train_data):>10,}")
print(f"  Preprocessing time  : {elapsed:>10.2f}s")
print("=" * 50)

## 3. SASRec Architecture

**Self-Attentive Sequential Recommendation** uses a stack of Transformer-style
blocks with causal (unidirectional) self-attention — no future leakage.

```
Input sequence  →  Item Emb + Pos Emb  →  [SASRec Block] × N  →  Final hidden  →  Dot-product score
```

Each **SASRec Block**:
1. `LayerNorm → MultiHeadAttention (causal) → Residual`
2. `LayerNorm → PointWiseFFN → Residual`

In [ ]:
class PointWiseFeedForward(nn.Module):
    """Position-wise feed-forward using 1-D convolutions."""

    def __init__(self, hidden_dim: int, dropout_rate: float):
        super().__init__()
        self.conv1    = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.conv2    = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.drop1    = nn.Dropout(dropout_rate)
        self.drop2    = nn.Dropout(dropout_rate)
        self.relu     = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)  →  transpose for Conv1d  →  (B, D, L)
        out = x.transpose(-1, -2)
        out = self.drop1(self.relu(self.conv1(out)))
        out = self.drop2(self.conv2(out))
        return out.transpose(-1, -2)  # NO RESIDUAL HERE


class SASRecBlock(nn.Module):
    """Single SASRec Transformer block."""

    def __init__(self, hidden_dim: int, num_heads: int, dropout_rate: float):
        super().__init__()
        self.attn   = nn.MultiheadAttention(hidden_dim, num_heads,
                                             dropout=dropout_rate,
                                             batch_first=True)
        self.ffn    = PointWiseFeedForward(hidden_dim, dropout_rate)
        self.norm1  = nn.LayerNorm(hidden_dim, eps=1e-8)
        self.norm2  = nn.LayerNorm(hidden_dim, eps=1e-8)
        self.drop   = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor,
                return_weights: bool = False):
        q = self.norm1(x)
        attn_out, attn_w = self.attn(q, q, q,
                                      attn_mask=causal_mask,
                                      need_weights=return_weights,
                                      is_causal=True)
        x = x + self.drop(attn_out)
        
        normed_x2 = self.norm2(x)
        ffn_out = self.ffn(normed_x2)
        x = x + self.drop(ffn_out)
        
        return (x, attn_w) if return_weights else x


class SASRec(nn.Module):
    """Self-Attentive Sequential Recommendation model."""

    def __init__(self, num_items: int, max_len: int,
                 hidden_dim: int = 64, num_blocks: int = 2,
                 num_heads: int = 1, dropout_rate: float = 0.2):
        super().__init__()
        self.item_emb    = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.pos_emb     = nn.Embedding(max_len, hidden_dim)
        self.emb_dropout = nn.Dropout(dropout_rate)
        self.blocks      = nn.ModuleList([
            SASRecBlock(hidden_dim, num_heads, dropout_rate)
            for _ in range(num_blocks)
        ])
        self.max_len    = max_len
        self.hidden_dim = hidden_dim
        self.num_items  = num_items
        self.num_blocks = num_blocks

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Embedding):
            # Embeddings often work best with small variance initialization in RS
            import torch.nn as nn
            nn.init.normal_(module.weight, mean=0.0, std=0.01)
        elif isinstance(module, nn.Linear) or isinstance(module, nn.Conv1d):
            import torch.nn as nn
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    @staticmethod
    def causal_mask(length: int, device) -> torch.Tensor:
        return torch.triu(torch.ones(length, length,
                                     dtype=torch.bool, device=device),
                          diagonal=1)

    def forward(self, log_seqs: torch.Tensor,
                return_weights: bool = False):
        B, L     = log_seqs.shape
        seqs     = self.item_emb(log_seqs)
        pos      = torch.arange(L, device=log_seqs.device).unsqueeze(0).expand(B, -1)
        seqs    += self.pos_emb(pos)
        seqs     = self.emb_dropout(seqs)
        pad_mask = (log_seqs == 0).unsqueeze(-1).float()
        mask     = self.causal_mask(L, log_seqs.device)
        all_weights = []
        for block in self.blocks:
            if return_weights:
                seqs, w = block(seqs, mask, return_weights=True)
                all_weights.append(w)
            else:
                seqs = block(seqs, mask)
            seqs = seqs * (1 - pad_mask)
        if return_weights:
            return seqs, all_weights
        return seqs

    def predict(self, log_seqs: torch.Tensor,
                item_indices: torch.Tensor) -> torch.Tensor:
        """Score all item_indices given log_seqs (batch)."""
        seq_out   = self.forward(log_seqs)
        final_out = seq_out[:, -1, :]          # (B, D)
        item_embs = self.item_emb(item_indices)  # (B, K, D)
        return torch.bmm(item_embs, final_out.unsqueeze(-1)).squeeze(-1)
\

## 4. Dataset & DataLoader

In [ ]:
class SASRecDataset(Dataset):
    """PyTorch Dataset with negative sampling for SASRec training."""

    def __init__(self, user_seqs: dict, max_len: int, num_items: int):
        self.user_seqs = user_seqs
        self.max_len   = max_len
        self.num_items = num_items
        self.users     = list(user_seqs.keys())

    def __len__(self) -> int:
        return len(self.users)

    def __getitem__(self, idx: int):
        seq    = self.user_seqs[self.users[idx]]
        tokens = seq[:-1][-self.max_len:]
        labels = seq[1:][-self.max_len:]
        pad    = self.max_len - len(tokens)
        tokens = [0] * pad + tokens
        labels = [0] * pad + labels
        user_set = set(seq)
        neg_labels = []
        for lbl in labels:
            if lbl == 0:
                neg_labels.append(0)
            else:
                n = random.randint(1, self.num_items - 1)
                while n in user_set:
                    n = random.randint(1, self.num_items - 1)
                neg_labels.append(n)
        return (torch.LongTensor(tokens),
                torch.LongTensor(labels),
                torch.LongTensor(neg_labels))

train_dataset = SASRecDataset(train_data, MAX_SEQ_LEN, NUM_ITEMS)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

print("=" * 50)
print("  DATASET SUMMARY")
print("=" * 50)
print(f"  Training samples    : {len(train_dataset):>10,}")
print(f"  Batches per epoch   : {len(train_loader):>10,}")
print(f"  Batch size          : {'256':>10}")
print(f"  Sequence length     : {MAX_SEQ_LEN:>10}")
print("=" * 50)

## 5. Training Loop

Trains for up to **100 epochs** with:
- 📊 **tqdm** progress bar showing live loss per batch
- ⏹️ **Early stopping** (patience = 15 epochs on Val NDCG@10)
- 💾 **Best-model saving** — restores the best checkpoint at the end


In [ ]:
# ── Install tqdm if needed ──────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'])
from tqdm.auto import tqdm

# ── Hyperparameters ──────────────────────────────────────────────
EPOCHS     = 100   # train up to 100 — early stopping will cut it short
PATIENCE   = 15    # stop if NDCG@10 doesn't improve for 15 epochs
LR         = 0.001
HIDDEN_DIM = 64
NUM_BLOCKS = 2
NUM_HEADS  = 1
DROPOUT    = 0.2

model     = SASRec(NUM_ITEMS, MAX_SEQ_LEN, HIDDEN_DIM, NUM_BLOCKS, NUM_HEADS, DROPOUT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98))
criterion = nn.BCEWithLogitsLoss(reduction="none")

# ── History + early-stopping state ───────────────────────────────
history = {
    "epoch":     [],
    "loss":      [],
    "val_hr":    [],
    "val_ndcg":  [],
    "test_hr":   None,
    "test_ndcg": None,
}
best_ndcg      = 0.0
best_state     = None
patience_count = 0
stopped_epoch  = EPOCHS

# ── Evaluation helper ─────────────────────────────────────────────
def evaluate(model, dataset_dict, train_dict, item_num, K=10):
    """Compute HR@K and NDCG@K via 99-negative sampling protocol."""
    model.eval()
    NDCG = HT = valid = 0.0
    with torch.no_grad():
        for u in dataset_dict:
            if not train_dict.get(u) or not dataset_dict.get(u):
                continue
            seq    = train_dict[u][-MAX_SEQ_LEN:]
            padded = [0] * (MAX_SEQ_LEN - len(seq)) + seq
            target = dataset_dict[u][0]
            user_set = set(train_dict[u])
            negs = []
            while len(negs) < 99:
                n = random.randint(1, item_num - 1)
                if n not in user_set and n != target:
                    negs.append(n)
            items = [target] + negs
            preds = model.predict(
                torch.LongTensor([padded]).to(DEVICE),
                torch.LongTensor([items]).to(DEVICE)
            )[0]
            rank = (preds > preds[0]).sum().item() + 1
            if rank <= K:
                HT   += 1
                NDCG += 1 / math.log2(rank + 1)
            valid += 1
    return HT / max(valid, 1), NDCG / max(valid, 1)

# ── Config banner ─────────────────────────────────────────────────
print("=" * 60)
print(f"  {'TRAINING SASREC MODEL':^56}")
print("=" * 60)
print(f"  {'Hyperparameter':<22} {'Value':>10}")
print(f"  {'-'*34}")
for k_, v_ in [('Max Epochs', EPOCHS), ('Early-Stop Patience', PATIENCE),
               ('Learning Rate', LR), ('Hidden Dim', HIDDEN_DIM),
               ('Num Blocks', NUM_BLOCKS), ('Num Heads', NUM_HEADS),
               ('Dropout Rate', DROPOUT), ('Max Seq Len', MAX_SEQ_LEN),
               ('Num Items', f'{NUM_ITEMS:,}')]:
    print(f"  {k_:<22} {str(v_):>10}")
print("=" * 60)

# ── Training loop ─────────────────────────────────────────────────
epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Training', unit='epoch',
                 bar_format='{l_bar}{bar:30}{r_bar}')

for epoch in epoch_bar:
    model.train()
    running_loss = 0.0

    # ── Batch bar (inner) ─────────────────────────────────────────
    batch_bar = tqdm(train_loader, desc=f'  Epoch {epoch:3d}', leave=False,
                     unit='batch',
                     bar_format='    {l_bar}{bar:25}{r_bar}')

    for seqs, pos, neg in batch_bar:
        seqs, pos, neg = seqs.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        optimizer.zero_grad()

        seq_emb    = model(seqs)
        pos_logits = (seq_emb * model.item_emb(pos)).sum(-1)
        neg_logits = (seq_emb * model.item_emb(neg)).sum(-1)
        target     = (pos > 0).float()

        loss = (
            (criterion(pos_logits, torch.ones_like(pos_logits)) +
             criterion(neg_logits, torch.zeros_like(neg_logits))) * target
        ).sum() / target.sum()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item()

        # Show live batch loss in the inner bar
        batch_bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────
    val_hr, val_ndcg = evaluate(model, val_data, train_data, NUM_ITEMS)

    history['epoch'].append(epoch)
    history['loss'].append(avg_loss)
    history['val_hr'].append(val_hr)
    history['val_ndcg'].append(val_ndcg)

    # ── Update outer bar ──────────────────────────────────────────
    star = ' ★' if val_ndcg > best_ndcg else '  '
    epoch_bar.set_postfix({
        'loss':    f'{avg_loss:.4f}',
        'HR@10':   f'{val_hr:.4f}',
        'NDCG@10': f'{val_ndcg:.4f}' + star,
    })

    # ── Early stopping ────────────────────────────────────────────
    if val_ndcg > best_ndcg:
        best_ndcg      = val_ndcg
        best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            stopped_epoch = epoch
            tqdm.write(f'\n  ⏹  Early stop at epoch {epoch}  |  Best NDCG@10 = {best_ndcg:.4f}')
            break

# ── Restore best weights ──────────────────────────────────────────
if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\n  ✅ Best model restored  |  NDCG@10 = {best_ndcg:.4f}  |  Stopped @ epoch {stopped_epoch}')

print("=" * 60)


## ✅ 6. Structured Results Display

Displays the training log, validation metrics, test metrics, and final
recommendations in a clean, labelled format.


In [ ]:
# ── Final test evaluation ─────────────────────────────────────────
train_val = {u: train_data[u] + val_data.get(u, []) for u in train_data}
test_hr, test_ndcg = evaluate(model, test_data, train_val, NUM_ITEMS)
history["test_hr"]   = test_hr
history["test_ndcg"] = test_ndcg

DIV  = "=" * 45
DIV2 = "-" * 45

# ── Training log ─────────────────────────────────────────────────
print(f"\n{DIV}")
print(f"  TRAINING LOG")
print(DIV)
print(f"  {'Epoch':<10} {'Loss':>12}")
print(f"  {DIV2}")
for ep, loss in zip(history['epoch'], history['loss']):
    print(f"  {f'{ep}/{EPOCHS}':<10} {loss:>12.4f}")
print(DIV)

# ── Validation metrics ────────────────────────────────────────────
print(f"\n{DIV}")
print(f"  VALIDATION METRICS  (Final Epoch)")
print(DIV)
print(f"  {'Metric':<25} {'Value':>10}")
print(f"  {DIV2}")
print(f"  {'HR@10':<25} {history['val_hr'][-1]:>10.4f}")
print(f"  {'NDCG@10':<25} {history['val_ndcg'][-1]:>10.4f}")
print(DIV)

# ── Test metrics ──────────────────────────────────────────────────
print(f"\n{DIV}")
print(f"  TEST METRICS")
print(DIV)
print(f"  {'Metric':<25} {'Value':>10}")
print(f"  {DIV2}")
print(f"  {'HR@10':<25} {test_hr:>10.4f}")
print(f"  {'NDCG@10':<25} {test_ndcg:>10.4f}")
print(DIV)

# ── Top-5 Recommendations for User 1 ─────────────────────────────
def get_top_k(user_id: int, k: int = 5):
    """Return top-K product IDs for a given user_id."""
    model.eval()
    if user_id not in user_sequences:
        return []
    seq    = user_sequences[user_id][-MAX_SEQ_LEN:]
    padded = [0] * (MAX_SEQ_LEN - len(seq)) + seq
    all_items = list(range(1, NUM_ITEMS))
    with torch.no_grad():
        preds = model.predict(
            torch.LongTensor([padded]).to(DEVICE),
            torch.LongTensor([all_items]).to(DEVICE)
        )[0].cpu().numpy()
    history_set = set(user_sequences[user_id])
    ranked = sorted(zip(all_items, preds), key=lambda x: x[1], reverse=True)
    recs = []
    for item_idx, _ in ranked:
        if item_idx not in history_set:
            recs.append(idx2item[item_idx])
        if len(recs) == k:
            break
    return recs

user_id = 1
recs    = get_top_k(user_id, k=5)

print(f"\n{DIV}")
print(f"  TOP 5 RECOMMENDATIONS  (User {user_id})")
print(DIV)
print(f"  {'Rank':<8} {'Product ID':>12}")
print(f"  {DIV2}")
for rank, pid in enumerate(recs, 1):
    print(f"  {rank:<8} {pid:>12}")
print(DIV)

## 📊 7. Visualisations & Model Statistics

Generates four publication-quality figures saved to `./figures/`:
1. **Training loss curve**
2. **Val HR@10 and NDCG@10 over epochs**
3. **Model architecture summary** (parameter breakdown table)
4. **Attention weight heatmap** (last SASRec block, first sample)

In [ ]:
PALETTE      = ["#6C63FF", "#FF6584", "#43D9AD", "#FFD166"]
ALPHA_FILL   = 0.15
LINEWIDTH    = 2.5

# ─────────────────────────────────────────────────────────────────
# Fig 1 — Training Loss Curve
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history["epoch"], history["loss"],
        color=PALETTE[0], lw=LINEWIDTH, marker="o", ms=6, label="Train Loss")
ax.fill_between(history["epoch"], history["loss"],
                alpha=ALPHA_FILL, color=PALETTE[0])
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("BCE Loss", fontsize=13)
ax.set_title("SASRec — Training Loss Curve", fontsize=15, fontweight="bold", pad=14)
ax.legend(fontsize=11)
ax.set_xticks(history["epoch"])
plt.tight_layout()
loss_path = os.path.join(FIGURES_DIR, "01_training_loss.png")
fig.savefig(loss_path, bbox_inches="tight")
plt.show()

print("=" * 45)
print("  FIGURE SAVED")
print("=" * 45)
print(f"  Path  : {os.path.abspath(loss_path)}")
print(f"  Size  : {os.path.getsize(loss_path) / 1024:.1f} KB")
print("=" * 45)

# ─────────────────────────────────────────────────────────────────
# Fig 2 — Evaluation Metrics (HR + NDCG) Over Epochs
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, label, color in zip(
    axes,
    ["val_hr", "val_ndcg"],
    ["HR@10", "NDCG@10"],
    [PALETTE[1], PALETTE[2]],
):
    ax.plot(history["epoch"], history[metric],
            color=color, lw=LINEWIDTH, marker="s", ms=6, label=f"Val {label}")
    ax.fill_between(history["epoch"], history[metric],
                    alpha=ALPHA_FILL, color=color)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f"Validation {label}", fontsize=14, fontweight="bold", pad=12)
    ax.set_xticks(history["epoch"])

    # Add final + test horizontal line
    final_val = history[metric][-1]
    test_val  = history["test_hr"] if metric == "val_hr" else history["test_ndcg"]
    ax.axhline(final_val, ls="--", color=color, alpha=0.5, label=f"Final Val = {final_val:.4f}")
    ax.axhline(test_val,  ls=":",  color="gray", alpha=0.7, label=f"Test = {test_val:.4f}")
    ax.legend(fontsize=9)

fig.suptitle("SASRec — Evaluation Metrics over Training", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
metrics_path = os.path.join(FIGURES_DIR, "02_eval_metrics.png")
fig.savefig(metrics_path, bbox_inches="tight")
plt.show()

print("=" * 45)
print("  FIGURE SAVED")
print("=" * 45)
print(f"  Path  : {os.path.abspath(metrics_path)}")
print(f"  Size  : {os.path.getsize(metrics_path) / 1024:.1f} KB")
print("=" * 45)

# ─────────────────────────────────────────────────────────────────
# Fig 3 — Model Architecture Summary (bar chart)
# ─────────────────────────────────────────────────────────────────
def count_params(module):
    return sum(p.numel() for p in module.parameters())

arch_rows = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Embedding, nn.MultiheadAttention,
                           nn.Conv1d, nn.LayerNorm, nn.Linear)):
        arch_rows.append({
            "Layer"  : name or "(root)",
            "Type"   : type(module).__name__,
            "Params" : count_params(module),
        })

arch_df = (
    pd.DataFrame(arch_rows)
    .groupby("Type", as_index=False)["Params"].sum()
    .sort_values("Params", ascending=False)
)

fig, (ax_bar, ax_table) = plt.subplots(1, 2, figsize=(14, 6),
                                        gridspec_kw={"width_ratios": [1.5, 1]})

bars = ax_bar.barh(arch_df["Type"], arch_df["Params"],
                   color=PALETTE[:len(arch_df)] * 10, edgecolor="white", height=0.6)
ax_bar.set_xlabel("Parameter Count", fontsize=12)
ax_bar.set_title("Parameters per Layer Type", fontsize=13, fontweight="bold")
for bar, val in zip(bars, arch_df["Params"]):
    ax_bar.text(bar.get_width() + max(arch_df["Params"]) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=10)

# Summary table
total_params     = count_params(model)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
table_data = [
    ["num_items",  f"{NUM_ITEMS:,}"],
    ["hidden_dim", str(HIDDEN_DIM)],
    ["num_blocks", str(NUM_BLOCKS)],
    ["num_heads",  str(NUM_HEADS)],
    ["max_seq_len",str(MAX_SEQ_LEN)],
    ["dropout",    str(DROPOUT)],
    ["─" * 12,     "─" * 12],
    ["Total Params",     f"{total_params:,}"],
    ["Trainable Params", f"{trainable_params:,}"],
]
ax_table.axis("off")
tbl = ax_table.table(
    cellText=table_data,
    colLabels=["Hyperparameter", "Value"],
    cellLoc="center", loc="center",
    colColours=["#6C63FF22"] * 2,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
ax_table.set_title("Model Configuration", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
arch_path = os.path.join(FIGURES_DIR, "03_model_architecture.png")
fig.savefig(arch_path, bbox_inches="tight")
plt.show()

print("=" * 50)
print("  MODEL ARCHITECTURE STATISTICS")
print("=" * 50)
print(f"  {'Hyperparameter':<22} {'Value':>12}")
print(f"  {'-'*36}")
for row in table_data:
    if row[0].startswith("─"):
        print(f"  {'-'*36}")
    else:
        print(f"  {row[0]:<22} {row[1]:>12}")
print("=" * 50)
print(f"  Figure saved : {os.path.abspath(arch_path)}")
print("=" * 50)

# ─────────────────────────────────────────────────────────────────
# Fig 4 — Attention Weight Heatmap (last block, first sample)
# ─────────────────────────────────────────────────────────────────
model.eval()
sample_user = list(train_data.keys())[0]
seq = train_data[sample_user][-MAX_SEQ_LEN:]
padded = [0] * (MAX_SEQ_LEN - len(seq)) + seq
seq_t  = torch.LongTensor([padded]).to(DEVICE)

with torch.no_grad():
    _, all_weights = model(seq_t, return_weights=True)

# Take the last block's attention weights  (1, L, L) → (L, L)
attn_w = all_weights[-1]
if attn_w is not None:
    attn_np = attn_w[0].cpu().numpy()  # (L, L)

    # Only show last 20 tokens to keep heatmap readable
    SHOW = 20
    attn_np = attn_np[-SHOW:, -SHOW:]

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(attn_np, ax=ax, cmap="magma_r",
                linewidths=0.3, linecolor="#ffffff22",
                vmin=0, xticklabels=False, yticklabels=False)
    ax.set_title(f"Attention Weight Heatmap — Last SASRec Block\n"
                 f"(User {sample_user}, last {SHOW} tokens)",
                 fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("Key (attended-to) position", fontsize=11)
    ax.set_ylabel("Query (current) position", fontsize=11)

    plt.tight_layout()
    attn_path = os.path.join(FIGURES_DIR, "04_attention_heatmap.png")
    fig.savefig(attn_path, bbox_inches="tight")
    plt.show()

    print("=" * 45)
    print("  ATTENTION HEATMAP SAVED")
    print("=" * 45)
    print(f"  Path  : {os.path.abspath(attn_path)}")
    print(f"  Shape : {attn_np.shape}")
    print("=" * 45)
else:
    print("[WARNING] Attention weights not available (avg_weights disabled).")

## 🔍 8. Inference / Prediction

Runs the trained model on a set of user IDs and displays structured output.


In [ ]:
TEST_USERS = [1, 2, 3]
TOP_K      = 5

print("=" * 55)
print(f"  {'RECOMMENDATION INFERENCE':^51}")
print("=" * 55)
print(f"  Top-K       : {TOP_K}")
print(f"  Users shown : {TEST_USERS}")
print("=" * 55)

for uid in TEST_USERS:
    recs = get_top_k(uid, k=TOP_K)
    print(f"\n  ── User {uid} ──────────────────────────────")
    if not recs:
        print(f"  [SKIP] No interaction history for user {uid}.")
        continue
    print(f"  {'Rank':<8} {'Product ID':>12}")
    print(f"  {'-'*22}")
    for rank, pid in enumerate(recs, 1):
        print(f"  {rank:<8} {pid:>12}")

print("\n" + "=" * 55)

## 💾 9. Model Export & Reload Test

Saves the full model checkpoint using `torch.save()`, then reloads it from disk
and runs an inference test to verify correctness.

The checkpoint includes:
- `model_state_dict`
- `item2idx` / `idx2item` mappings
- `user_sequences`
- `hyperparams`
- `metrics` (val + test HR/NDCG)

In [ ]:
CHECKPOINT_DIR  = os.path.join(BASE_PATH, "api", "checkpoints")
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "sasrec_emall.pth")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Build the checkpoint dict ────────────────────────────────────
checkpoint = {
    "model_state_dict": model.state_dict(),
    "item2idx"        : item2idx,
    "idx2item"        : idx2item,
    "user_sequences"  : user_sequences,
    "hyperparams"     : {
        "num_items"   : NUM_ITEMS,
        "max_len"     : MAX_SEQ_LEN,
        "hidden_dim"  : HIDDEN_DIM,
        "num_blocks"  : NUM_BLOCKS,
        "num_heads"   : NUM_HEADS,
        "dropout_rate": DROPOUT,
    },
    "metrics": {
        "val_hr_at_10"       : history["val_hr"][-1],
        "val_ndcg_at_10"     : history["val_ndcg"][-1],
        "test_hr_at_10"      : history["test_hr"],
        "test_ndcg_at_10"    : history["test_ndcg"],
    },
}

# ── Save ─────────────────────────────────────────────────────────
t0 = time.time()
torch.save(checkpoint, CHECKPOINT_PATH)
save_time   = time.time() - t0
file_size_b = os.path.getsize(CHECKPOINT_PATH)

print("=" * 55)
print("  MODEL SAVED")
print("=" * 55)
print(f"  {'Save path':<22} : {os.path.abspath(CHECKPOINT_PATH)}")
print(f"  {'File size':<22} : {file_size_b / (1024**2):.2f} MB  ({file_size_b:,} bytes)")
print(f"  {'Save time':<22} : {save_time:.3f}s")
print("=" * 55)

In [ ]:
# ── Reload & Inference Test ──────────────────────────────────────
print("=" * 55)
print("  RELOAD + INFERENCE TEST")
print("=" * 55)

t0         = time.time()
ckpt       = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
hp         = ckpt["hyperparams"]
load_time  = time.time() - t0

model_reload = SASRec(
    num_items    = hp["num_items"],
    max_len      = hp["max_len"],
    hidden_dim   = hp["hidden_dim"],
    num_blocks   = hp["num_blocks"],
    num_heads    = hp["num_heads"],
    dropout_rate = hp["dropout_rate"],
).to(DEVICE)

model_reload.load_state_dict(ckpt["model_state_dict"])
model_reload.eval()

r_item2idx = ckpt["item2idx"]
r_idx2item = ckpt["idx2item"]
r_user_seqs = ckpt["user_sequences"]
r_metrics   = ckpt["metrics"]

print(f"  {'Load path':<22} : {os.path.abspath(CHECKPOINT_PATH)}")
print(f"  {'Load time':<22} : {load_time:.3f}s")
print(f"  {'Device':<22} : {DEVICE}")
print(f"  {'num_items':<22} : {hp['num_items']:,}")
print(f"  {'hidden_dim':<22} : {hp['hidden_dim']}")
print(f"  {'num_blocks':<22} : {hp['num_blocks']}")
print("-" * 55)
print(f"  {'Stored Metrics'}")
print(f"  {'Val HR@10':<22} : {r_metrics['val_hr_at_10']:.4f}")
print(f"  {'Val NDCG@10':<22} : {r_metrics['val_ndcg_at_10']:.4f}")
print(f"  {'Test HR@10':<22} : {r_metrics['test_hr_at_10']:.4f}")
print(f"  {'Test NDCG@10':<22} : {r_metrics['test_ndcg_at_10']:.4f}")

# ── Run inference with reloaded model ────────────────────────────
test_uid = 1
if test_uid in r_user_seqs:
    r_seq    = r_user_seqs[test_uid][-MAX_SEQ_LEN:]
    r_padded = [0] * (MAX_SEQ_LEN - len(r_seq)) + r_seq
    r_items  = list(range(1, hp["num_items"]))
    with torch.no_grad():
        r_preds = model_reload.predict(
            torch.LongTensor([r_padded]).to(DEVICE),
            torch.LongTensor([r_items]).to(DEVICE)
        )[0].cpu().numpy()
    r_ranked = sorted(zip(r_items, r_preds), key=lambda x: x[1], reverse=True)
    r_recs   = [r_idx2item[i] for i, _ in r_ranked if i not in set(r_user_seqs[test_uid])][:5]

    print("-" * 55)
    print(f"  Reloaded-model Top-5 for User {test_uid}")
    print(f"  {'Rank':<8} {'Product ID':>12}")
    print(f"  {'-'*22}")
    for rank, pid in enumerate(r_recs, 1):
        print(f"  {rank:<8} {pid:>12}")

print("=" * 55)
print("  [OK] Reload + inference test PASSED")
print("=" * 55)

## 🌐 10. Serve AI API from Colab (Postman / curl ready)

Because Colab has no public IP, we use **pyngrok** to expose the FastAPI server.

### Steps
1. Create a free account at https://dashboard.ngrok.com/
2. Copy your **Authtoken** (Settings page)
3. Paste it below where it says `YOUR_NGROK_TOKEN`
4. Run the cell — a `https://xxxx.ngrok-free.app` URL will appear
5. Use that URL as the **Base URL** in Postman

> Keep the cell running — stopping it shuts the server down.

In [ ]:
import subprocess, sys, threading, time, inspect

def _pip(*p): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *p])
_pip('fastapi', 'uvicorn[standard]', 'pyngrok')

# === PASTE YOUR NGROK TOKEN HERE ===
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Build self-contained API file so uvicorn can import it
_header = (
    'import torch, torch.nn as nn\n'
    'from fastapi import FastAPI, HTTPException\n'
    'from fastapi.middleware.cors import CORSMiddleware\n'
    'from pydantic import BaseModel, Field\n'
    'from typing import List\n'
)
_model_src = '\n\n'.join([
    inspect.getsource(PointWiseFeedForward),
    inspect.getsource(SASRecBlock),
    inspect.getsource(SASRec),
])
_body = f'''
CKPT='{CHECKPOINT_PATH}'
MAX_SEQ_LEN=50
ck=torch.load(CKPT,map_location='cpu',weights_only=False)
hp=ck['hyperparams']
_m=SASRec(hp['num_items'],hp['max_len'],hp['hidden_dim'],
          hp['num_blocks'],hp['num_heads'],hp['dropout_rate'])
_m.load_state_dict(ck['model_state_dict'])
_m.eval()
i2i=ck['item2idx']; i2p=ck['idx2item']
useqs=ck['user_sequences']; N=hp['num_items']
met=ck.get('metrics',{{}})

app=FastAPI(title='E-Mall SASRec API',version='1.0.0')
app.add_middleware(CORSMiddleware,allow_origins=['*'],
                   allow_methods=['*'],allow_headers=['*'])

class RR(BaseModel):
    user_id:int=Field(...,ge=1)
    top_k:int=Field(10,ge=1,le=50)
    exclude_interacted:bool=True

class SR(BaseModel):
    product_ids:List[int]
    top_k:int=Field(10,ge=1,le=50)

class MR(BaseModel):
    product_id:int=Field(...,ge=1)
    top_k:int=Field(10,ge=1,le=50)

def _rank(seq):
    items=list(range(1,N))
    pad=[0]*(MAX_SEQ_LEN-len(seq))+seq[-MAX_SEQ_LEN:]
    import torch
    with torch.no_grad():
        sc=_m.predict(torch.LongTensor([pad]),torch.LongTensor([items]))[0].numpy()
    return sorted(zip(items,sc),key=lambda x:x[1],reverse=True)

@app.get('/health')
def health(): return {{'status':'ok','num_items':N-1,'num_users':len(useqs),'metrics':met}}

@app.post('/recommend')
def recommend(r:RR):
    if r.user_id not in useqs: raise HTTPException(404,f'User {{r.user_id}} not found')
    excl=set(useqs[r.user_id]) if r.exclude_interacted else set()
    out=[]
    for idx,sc in _rank(useqs[r.user_id]):
        if idx in excl: continue
        out.append({{'rank':len(out)+1,'product_id':i2p.get(idx,idx),'score':round(float(sc),4)}})
        if len(out)>=r.top_k: break
    return {{'user_id':r.user_id,'recommendations':out}}

@app.post('/recommend/sequence')
def rec_seq(r:SR):
    seq=[i2i[p] for p in r.product_ids if p in i2i]
    if not seq: raise HTTPException(400,'No known product_ids')
    excl,out=set(seq),[]
    for idx,sc in _rank(seq):
        if idx in excl: continue
        out.append({{'rank':len(out)+1,'product_id':i2p.get(idx,idx),'score':round(float(sc),4)}})
        if len(out)>=r.top_k: break
    return {{'recommendations':out}}

@app.post('/recommend/similar')
def similar(r:MR):
    if r.product_id not in i2i: raise HTTPException(404,f'Product {{r.product_id}} not known')
    tidx=i2i[r.product_id]
    import torch
    with torch.no_grad():
        embs=_m.item_emb.weight.data; t=embs[tidx].unsqueeze(0)
        sims=((embs/embs.norm(dim=-1,keepdim=True).clamp(1e-8)) @
              (t/t.norm(dim=-1,keepdim=True).clamp(1e-8)).T).squeeze(-1).numpy()
    pairs=sorted([(i,float(sims[i])) for i in range(1,len(sims)) if i!=tidx],
                 key=lambda x:x[1],reverse=True)
    return {{'product_id':r.product_id,
            'similar_items':[{{'rank':k+1,'product_id':i2p.get(i,i),'similarity':round(s,4)}}
                             for k,(i,s) in enumerate(pairs[:r.top_k])]}}
'''

with open('/content/colab_api.py','w') as fh:
    fh.write(_header+'\n\n'+_model_src+'\n\n'+_body)

def _run():
    import uvicorn
    uvicorn.run('colab_api:app',host='0.0.0.0',port=8000,log_level='warning')

threading.Thread(target=_run,daemon=True).start()
time.sleep(4)

url=ngrok.connect(8000).public_url
print('='*56)
print('  E-Mall SASRec API  -  LIVE')
print('='*56)
print(f'  Base URL   : {url}')
print(f'  Swagger UI : {url}/docs')
print(f'  ReDoc      : {url}/redoc')
print('='*56)
print(f'  GET  {url}/health')
print(f'  POST {url}/recommend')
print('       body: {"user_id":1, "top_k":5}')
print(f'  POST {url}/recommend/sequence')
print('       body: {"product_ids":[321,705,14], "top_k":5}')
print(f'  POST {url}/recommend/similar')
print('       body: {"product_id":321, "top_k":5}')
print('='*56)